# 04 — Color-Color Plots

Construct color-color diagrams to visualize the UV-excess candidate selection made in
`01_crossmatch.ipynb`. Two views:

1. **Candidates only** (NUV-G vs G-R, mag differences, colored by E(B-V)) — for the already-selected
   `uv_excess_candidates.csv`. Source: `scripts/colorcolor/colorcolor.py`.
2. **Full combined sample** (NUV/G vs G/R and FUV/NUV vs NUV/G, flux ratios, dual-axis) — for
   exploring the whole `FINAL_COMBINED_QSOs_W2M.csv` sample, not just candidates. Source:
   `scripts/colorcolor/colorcolor_w2m_gri.py`.

**This notebook visualizes; it does not (re)compute the UV-excess selection** — that selection is
canonical in `01_crossmatch.ipynb`'s candidate-CSV section. Re-run that notebook first if the
crossmatch outputs have changed.

**Inputs:** `data/matched/uv_excess_candidates.csv`, `data/matched/FINAL_COMBINED_QSOs_W2M.csv`
**Script references:** `scripts/colorcolor/colorcolor.py`, `colorcolor_w2m_gri.py`

Run `/review-uv-excess` after this notebook to summarize the candidate sample.

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import yaml
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy.units as u
from synphot import units as su

BASE_DIR = Path.cwd().parent
with open(BASE_DIR / "config" / "qso_params.yaml") as f:
    PARAMS = yaml.safe_load(f)

DATA_MATCHED = BASE_DIR / "data" / "matched"
WAVELENGTHS = PARAMS["band_wavelengths_AA"]
FLAM_MAX_GALEX_PS1 = PARAMS["flux_outlier_thresholds_flam"]["galex_panstarrs"] * su.FLAM
TARGET_SAMPLE_MAX = PARAMS["uv_excess"]["target_sample_max"]


def mag_arr(col):
    if hasattr(col, 'filled'):
        return np.array(col.filled(np.nan), dtype=float)
    arr = np.array(col, dtype=str)
    arr[arr == '--'] = 'nan'
    return arr.astype(float)


def to_flam(mags, band):
    lam = WAVELENGTHS[band] * u.AA
    flam = (mags * u.ABmag).to(su.FLAM, u.spectral_density(lam))
    flam[flam > FLAM_MAX_GALEX_PS1] = np.nan
    return flam

## Candidates only: NUV-G vs G-R (magnitude differences)

Source: `scripts/colorcolor/colorcolor.py`. Colored by E(B-V); the axhline/axvline at 0 mark the
UV-excess quadrant boundary.

In [ ]:
cand_table = Table.read(DATA_MATCHED / "uv_excess_candidates.csv", format='csv')

redshift = cand_table['Z']
ebv = cand_table['EBV']

fuv_mags = mag_arr(cand_table['FUVmag'])
nuv_mags = mag_arr(cand_table['NUVmag'])
g_mags = mag_arr(cand_table['gmag'])
r_mags = mag_arr(cand_table['rmag'])

nuvg = nuv_mags - g_mags
gr = g_mags - r_mags

plt.figure(figsize=(10, 10))
sc = plt.scatter(nuvg, gr, c=redshift, cmap='inferno', s=100)
plt.title(f'Color vs Color for {len(cand_table)} QSOs with UV Excess')
plt.colorbar(label='E(B-V)')
plt.axhline(y=0)
plt.axvline(x=0)
plt.xlabel('NUV-G')
plt.ylabel('G-R')
plt.show()

## Full combined sample: NUV/G vs G/R and FUV/NUV vs NUV/G (flux ratios)

Source: `scripts/colorcolor/colorcolor_w2m_gri.py`. Explores the entire DESI+W2M combined
sample (not just pre-selected candidates), using flux ratios on a dual blue/red axis. The
axhline/axvline at 1 mark the UV-excess boundary (a ratio > 1 means the bluer band dominates).

In [ ]:
full_table = Table.read(DATA_MATCHED / "FINAL_COMBINED_QSOs_W2M.csv", format='csv')

FUVmags = mag_arr(full_table['FUVmag'])
NUVmags = mag_arr(full_table['NUVmag'])
gmags = mag_arr(full_table['gmag'])
rmags = mag_arr(full_table['rmag'])
imags = mag_arr(full_table['imag'])

flam_fuv = to_flam(FUVmags, 'FUV')
flam_nuv = to_flam(NUVmags, 'NUV')
flam_g = to_flam(gmags, 'g')
flam_r = to_flam(rmags, 'r')
flam_i = to_flam(imags, 'i')

fuvnuv = flam_fuv / flam_nuv
nuvg = flam_nuv / flam_g
gr = flam_g / flam_r
ri = flam_r / flam_i

# mask out rows where any underlying mag used in these colors is 0
# (0 indicates a missing/bad measurement, not a real magnitude)
zero_mask = (FUVmags == 0) | (NUVmags == 0) | (gmags == 0) | (rmags == 0) | (imags == 0)
fuvnuv[zero_mask] = np.nan
nuvg[zero_mask] = np.nan
gr[zero_mask] = np.nan
ri[zero_mask] = np.nan

fig, ax_blue = plt.subplots(figsize=(10, 10))
ax_red = ax_blue.twiny().twinx()

sc_blue = ax_blue.scatter(nuvg, fuvnuv, c='blue', s=100, zorder=2, alpha=0.5, label='FUV/NUV vs NUV/G')
sc_red = ax_red.scatter(gr, nuvg, c='red', s=100, zorder=2, alpha=0.5, label='NUV/G vs G/R')

ax_blue.set_xlabel('NUV/G (blue)', color='blue')
ax_blue.set_ylabel('FUV/NUV (blue)', color='blue')
ax_blue.tick_params(axis='x', colors='blue')
ax_blue.tick_params(axis='y', colors='blue')

ax_red.xaxis.set_label_position('top')
ax_red.xaxis.tick_top()
ax_red.set_xlabel('G/R (red)', color='red')
ax_red.set_ylabel('NUV/G (red)', color='red')
ax_red.tick_params(axis='x', colors='red')
ax_red.tick_params(axis='y', colors='red')

ax_blue.axhline(y=1)
ax_blue.axvline(x=1)

handles = [sc_blue, sc_red]
labels = [h.get_label() for h in handles]
ax_blue.legend(handles, labels, loc='best')

plt.title(f'Full combined sample (N={len(full_table)})')
plt.tight_layout()
plt.show()